# Bathymetry — catalog explorer (offline)

Inspect the bundled DEM catalog without touching the network: which DEMs
ship, their resolutions, the elevation band each exposes, the ERDDAP
`griddap` endpoint behind each, and the licence note. This is the no-network
companion to the live quickstart.

In [ ]:
import pandas as pd

from earthlens.bathymetry import Catalog

## The shipped DEMs

`Catalog()` loads the sharded `catalog/` directory (a per-family YAML per
source plus an `_index.yaml`). Each row is a frozen pydantic `Dataset`.

In [ ]:
catalog = Catalog()
rows = [
    {
        'id': d,
        'resolution': catalog.get(d).native_resolution,
        'band': catalog.get(d).variable,
        'lon': catalog.get(d).lon_convention,
        'transport': catalog.get(d).transport,
    }
    for d in sorted(catalog.datasets)
]
pd.DataFrame(rows).set_index('id')

## Endpoints and attribution

Every DEM is one uniform NOAA ERDDAP `griddap` coverage. The `endpoint` +
`dataset_id` are what the backend turns into a bbox-subset `.nc` URL.

In [ ]:
for d in sorted(catalog.datasets):
    row = catalog.get(d)
    print(f'{d}\n  {row.endpoint}/griddap/{row.dataset_id}\n  {row.license_note}\n')

## Build a request URL

The same helper the backend uses turns a row + bbox into the exact `griddap`
`.nc` subset URL — handy for inspecting or fetching by hand.

In [ ]:
from earthlens.bathymetry._helpers import griddap_subset_url

row = catalog.get('gebco_2020')
griddap_subset_url(
    row.endpoint, row.dataset_id, row.variable, (-18.0, 25.0, -17.0, 26.0)
)